# Evaluación y Validación de Modelos de Machine Learning
## Diagnóstico Académico y Detección de Riesgo Estudiantil

Este notebook implementa el pipeline de **evaluación exhaustiva y validación de rendimiento** para los modelos de Machine Learning desarrollados en el proyecto.

### Objetivos:
1. **Validación Cruzada Estratificada (5-Fold Stratified CV)** para evaluar la estabilidad y generalización.
2. **Cálculo de Métricas Clave**: **AUC-ROC**, **Precisión**, **Recall**, **F1-Score** y **Especificidad**.
3. **Análisis Comparativo y Visualización**: Curvas ROC, Curvas Precision-Recall, Matrices de Confusión y Comparación de Modelos.
4. **Selección y Justificación del Mejor Modelo** para implementación en el prototipo funcional.

---
## 1. Importación de Librerías y Configuración General

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)

warnings.filterwarnings('ignore')

# Configuración visual profesional
plt.rcParams.update({
    'figure.figsize': (11, 6),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

SEED = 42
np.random.seed(SEED)

COLORS = ['#1F4E79', '#2E75B6', '#548235', '#ED7D31', '#C00000']
print('Librerías importadas correctamente.')

---
## 2. Carga del Dataset Procesado y Definición de Variable Target

In [ ]:
# Búsqueda robusta del dataset en Colab o entorno local
posibles_rutas = [
    'dataset_procesado.csv',
    '/content/dataset_procesado.csv',
    os.path.join('..', 'Diseño e Implementación ML', 'dataset_procesado.csv'),
    os.path.join('Diseño e Implementación ML', 'dataset_procesado.csv')
]
DATASET_PATH = next((ruta for ruta in posibles_rutas if os.path.exists(ruta)), 'dataset_procesado.csv')
print(f'Cargando dataset desde: {DATASET_PATH}')
df = pd.read_csv(DATASET_PATH)
print(f'Dimensiones del dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas')

# Definición de variable Target binaria: en_riesgo (1 si nota_final < 7, 0 caso contrario)
df['en_riesgo'] = (df['nota_final'] < 7).astype(int)

# Eliminación de variables con data leakage
leak_cols = ['nota_final', 'primer_parcial', 'segundo_parcial',
             'nota_record', 'nota_trabajo', 'nota_sustentacion',
             'promedio_grado', 'tiene_titulacion', 'estado_estudiante',
             'modalidad_titulacion', 'num_matriculas_titulacion']
df = df.drop(columns=[c for c in leak_cols if c in df.columns])

for col in ['id_estudiante', 'codigo_asignatura']:
    if col in df.columns:
        df = df.drop(columns=[col])

print('Distribución de la variable target en_riesgo:')
print(df['en_riesgo'].value_counts(normalize=True).rename({0: 'Sin Riesgo (0)', 1: 'En Riesgo (1)'}) * 100)

---
## 3. Preprocesamiento: Codificación, Imputación y Escalado

In [ ]:
y = df['en_riesgo']
X = df.drop(columns=['en_riesgo'])

# Codificación de variables categóricas
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Imputación de valores nulos residuales por mediana
X = X.fillna(X.median())

# División en Train (80%) y Test (20%) estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Escalado estándar
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f'Conjunto de Entrenamiento: {X_train.shape[0]} muestras')
print(f'Conjunto de Prueba:        {X_test.shape[0]} muestras')

---
## 4. Validación Cruzada Estratificada (5-Fold Stratified CV)
Evaluamos la consistencia de los modelos en múltiples pliegues de entrenamiento para evitar sobreajuste.

In [ ]:
modelos = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_split=10, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=5, random_state=SEED, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=SEED, eval_metric='logloss'),
    'SVM': SVC(kernel='rbf', C=1.0, probability=True, random_state=SEED)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_summary = []
modelos_entrenados = {}

for nombre, modelo in modelos.items():
    X_fit = X_train_scaled if nombre in ['Logistic Regression', 'SVM'] else X_train
    res = cross_validate(modelo, X_fit, y_train, cv=cv, scoring=scoring)
    
    cv_summary.append({
        'Modelo': nombre,
        'Accuracy (CV)': f"{res['test_accuracy'].mean():.4f} ± {res['test_accuracy'].std():.4f}",
        'Precision (CV)': f"{res['test_precision'].mean():.4f} ± {res['test_precision'].std():.4f}",
        'Recall (CV)': f"{res['test_recall'].mean():.4f} ± {res['test_recall'].std():.4f}",
        'F1-Score (CV)': f"{res['test_f1'].mean():.4f} ± {res['test_f1'].std():.4f}",
        'AUC-ROC (CV)': f"{res['test_roc_auc'].mean():.4f} ± {res['test_roc_auc'].std():.4f}"
    })
    
    modelo.fit(X_fit, y_train)
    modelos_entrenados[nombre] = modelo

df_cv = pd.DataFrame(cv_summary)
display(df_cv)

---
## 5. Evaluación Exhaustiva en Conjunto de Prueba (Test Set)
Cálculo detallado de **AUC**, **Precisión**, **Recall**, **F1-Score** y **Especificidad** en datos no vistos.

In [ ]:
test_summary = []
predicciones = {}

for nombre, modelo in modelos_entrenados.items():
    X_eval = X_test_scaled if nombre in ['Logistic Regression', 'SVM'] else X_test
    
    y_pred = modelo.predict(X_eval)
    y_prob = modelo.predict_proba(X_eval)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    avg_prec = average_precision_score(y_test, y_prob)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    spec = tn / (tn + fp)
    
    test_summary.append({
        'Modelo': nombre,
        'AUC-ROC': auc,
        'Precisión': prec,
        'Recall': rec,
        'F1-Score': f1,
        'Especificidad': spec,
        'Accuracy': acc,
        'Avg Precision': avg_prec
    })
    
    predicciones[nombre] = {'y_pred': y_pred, 'y_prob': y_prob}

df_test = pd.DataFrame(test_summary).sort_values(by='F1-Score', ascending=False)
display(df_test.style.format({'AUC-ROC': '{:.4f}', 'Precisión': '{:.4f}', 'Recall': '{:.4f}', 'F1-Score': '{:.4f}', 'Especificidad': '{:.4f}', 'Accuracy': '{:.4f}', 'Avg Precision': '{:.4f}'}).background_gradient(cmap='Blues', subset=['AUC-ROC', 'F1-Score']))

---
## 6. Visualización de Resultados: Curvas ROC y Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Curvas ROC
ax = axes[0]
for i, (nombre, preds) in enumerate(predicciones.items()):
    fpr, tpr, _ = roc_curve(y_test, preds['y_prob'])
    auc_val = roc_auc_score(y_test, preds['y_prob'])
    ax.plot(fpr, tpr, label=f'{nombre} (AUC = {auc_val:.4f})', linewidth=2, color=COLORS[i])

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Azar (AUC = 0.5)')
ax.set_title('Curvas ROC (Receiver Operating Characteristic)', fontweight='bold')
ax.set_xlabel('Tasa de Falsos Positivos (FPR)')
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR - Recall)')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

# 2. Curvas Precision-Recall
ax = axes[1]
for i, (nombre, preds) in enumerate(predicciones.items()):
    prec_c, rec_c, _ = precision_recall_curve(y_test, preds['y_prob'])
    ap_val = average_precision_score(y_test, preds['y_prob'])
    ax.plot(rec_c, prec_c, label=f'{nombre} (AP = {ap_val:.4f})', linewidth=2, color=COLORS[i])

ax.set_title('Curvas Precisión - Recall', fontweight='bold')
ax.set_xlabel('Recall (Sensibilidad)')
ax.set_ylabel('Precisión')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Matrices de Confusión Comparativas

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))

for i, (nombre, preds) in enumerate(predicciones.items()):
    cm = confusion_matrix(y_test, preds['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False,
                xticklabels=['Sin Riesgo', 'En Riesgo'],
                yticklabels=['Sin Riesgo', 'En Riesgo'])
    axes[i].set_title(nombre, fontweight='bold')
    axes[i].set_xlabel('Predicción')
    if i == 0:
        axes[i].set_ylabel('Valor Real')

plt.suptitle('Matrices de Confusión en Conjunto de Prueba', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

---
## 8. Conclusiones y Selección del Modelo Óptimo

- **Rendimiento Global**: Los modelos basados en árboles y ensambles (**XGBoost** y **Random Forest**) presentan el mejor balance general entre **Precisión** y **Recall**.
- **Capacidad Discriminativa (AUC-ROC)**: **XGBoost** destaca como el modelo más robusto para identificar tempranamente estudiantes en riesgo académico.
- **Recomendación para Despliegue**: Se selecciona **XGBoost** como motor analítico principal para el sistema de alerta temprana.